# Lesson 08 Lab — Inside an L2 Slice: Tags, Banks, and Miss State

**Puzzle:** An L2 cache is made from SRAM, so why is its behavior more than just a fast array?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A cache slice combines tag arrays, comparators, data banks, replacement state, queues, and miss-status tracking. An address is split into offset, set, and tag; tags decide whether a line is present, banks provide parallel access, and MSHR-like state tracks outstanding misses until refill. Conflicts can arise from placement, ports, banks, queues, or downstream memory even when capacity looks sufficient.


## 0. Predict before running

1. Split a sample byte address into offset, set, and tag.
2. Predict requested bandwidth as stride grows.
3. Explain why this experiment cannot directly report L2 hit rate.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The lab sweeps a large CUDA tensor with contiguous and increasingly sparse strides. It reports requested bandwidth per accessed element. This is a locality probe, not a direct L2-hit counter: changing stride changes useful bytes per transaction and cache-line reuse, while reduction work and compiler kernels remain involved. The conceptual diagram labels likely components without claiming a die-accurate NVIDIA implementation.

- Tag lookup and data access are distinct operations.
- Banking increases service parallelism but does not remove finite ports or queues.
- A miss consumes state until refill; too many outstanding misses can backpressure requesters.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["request address"] --> B["tag + set + offset"]
  B --> C["tag compare"]
  C -->|"hit"| D["banked data array"]
  C -->|"miss"| E["miss-status entry"]
  E --> F["memory refill"]
  F --> D
```


## 3. Inspect the visual boundary

![Conceptual L2 slice](../assets/L2_cache_slice_circuit_structure.png)

- [Printable L2 slice diagram](../assets/L2_cache_slice_circuit_structure_A4_portrait.pdf)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 8
LESSON_TITLE = 'Inside an L2 Slice: Tags, Banks, and Miss State'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260821
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | contiguous access over the tensor |
| Candidate | strides 2, 4, 8, 16, and 32 |
| Held constant | source allocation, dtype, accessed-element accounting, and timing helper |
| Measurements | median latency, requested GB/s, address tag/set/offset |
| Evidence | `pytorch-gpu` |

**Experiment:** Sweep CUDA access stride and retain a transparent address-decomposition example.


## 6. Inspect the code

Each view is consumed by the same reduction expression. The code counts only useful values, prints the decomposition assumptions, and labels the result as a PyTorch locality probe rather than a cache-counter measurement.

Do not run until the code matches the frozen table.


In [2]:
n = 2**26
x = torch.randn(n, device=DEVICE, dtype=torch.float32)
rows = {}
for stride in (1, 2, 4, 8, 16, 32):
    view = x[::stride]
    samples = cuda_samples(lambda view=view: (view * 1.0001).sum(), warmup=3, repeats=15)
    median = statistics.median(samples)
    useful_bytes = view.numel() * view.element_size()
    rows[f"stride_{stride}"] = {
        "elements": view.numel(), "median_ms": median,
        "requested_gbps": useful_bytes / (median / 1e3) / 1e9,
        "samples_ms": samples,
    }

line_bytes, sets = 128, 4096
address = 0x1234ABCD
offset_bits = int(math.log2(line_bytes)); set_bits = int(math.log2(sets))
address_example = {
    "address_hex": hex(address),
    "offset": address & (line_bytes - 1),
    "set": (address >> offset_bits) & (sets - 1),
    "tag": address >> (offset_bits + set_bits),
    "assumed_line_bytes": line_bytes, "assumed_sets": sets,
}
metrics = {**rows, "address_example": address_example}
analysis = (
    f"Requested bandwidth fell from {rows['stride_1']['requested_gbps']:.1f} GB/s at stride 1 "
    f"to {rows['stride_32']['requested_gbps']:.1f} GB/s at stride 32. This is a locality probe; "
    "no L2 hit-rate counter was collected."
)
print(json.dumps(metrics, indent=2))


{
  "stride_1": {
    "elements": 67108864,
    "median_ms": 0.5062400102615356,
    "requested_gbps": 530.2533394413449,
    "samples_ms": [
      0.5145279765129089,
      0.5084159970283508,
      0.5052800178527832,
      0.50627201795578,
      0.5050240159034729,
      0.5062400102615356,
      0.5066879987716675,
      0.5069119930267334,
      0.5056319832801819,
      0.505728006362915,
      0.5084480047225952,
      0.5056319832801819,
      0.50518399477005,
      0.5054399967193604,
      0.5071679949760437
    ]
  },
  "stride_2": {
    "elements": 33554432,
    "median_ms": 0.3080640137195587,
    "requested_gbps": 435.6812935709622,
    "samples_ms": [
      0.30777600407600403,
      0.30902400612831116,
      0.3080640137195587,
      0.3072960078716278,
      0.30799999833106995,
      0.30793601274490356,
      0.3101760149002075,
      0.30831998586654663,
      0.3104960024356842,
      0.3060159981250763,
      0.30799999833106995,
      0.30777600407600403,
    

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Contiguous median | 0.506 ms |
| Contiguous requested bandwidth | 530.2533 |
| Stride-8 requested bandwidth | 183.9607 |
| Stride-32 requested bandwidth | 128.8810 |
| Example cache set | 2,391 |


## 8. Explain rather than overclaim

Requested bandwidth fell from 530.3 GB/s at stride 1 to 128.9 GB/s at stride 32. This is a locality probe; no L2 hit-rate counter was collected.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 8, "title": 'Inside an L2 Slice: Tags, Banks, and Miss State', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Use stride timing to form a cache hypothesis, then require hardware counters before attributing the result to L2 hits, sectors, or miss queues.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 8,
  "title": "Inside an L2 Slice: Tags, Banks, and Miss State",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260821
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "stride_1": {
      "elements": 67108864,
      "median_ms": 0.5062400102615356,
      "requested_gbps": 530.2533394413449,
      "samples_ms": [
        0.5145279765129089,
        0.5084159970283508,
        0.5052800178527832,
        0.50627201795578,
        0.5050240159034729,
        0.5062400102615356,
        0.5066879987716675,
        0.5069119930267334,
        0.5056319832801819,
        0.505728006362915,
        0.5084480047225952,
        0.5056319832801819,
        0.50518399477005,
        0.5054399967193604,
        0.5071679949760437
      ]
    },
    "stride_2": {
      "elements": 33554432,
      "median_ms": 0.3080640137195587,
      "re

## 10. Make the decision

> Use stride timing to form a cache hypothesis, then require hardware counters before attributing the result to L2 hits, sectors, or miss queues.

**Failure analysis:** Reduction scheduling and lower element count at large stride complicate comparisons. Prefetching, cache state, and clock variation can also alter the curve.


## 11. Extend the evidence

Profile L2 sectors, hit rate, and DRAM bytes for equal-work custom kernels while sweeping working-set size across cache capacity.

See [`README.md`](README.md) for the full explanation and references.
